# Feature Extraction: TF-IDF and Text Features

Demonstrates text feature extraction techniques, specifically **TF-IDF (Term Frequency-Inverse Document Frequency)**, a fundamental method for converting text into numerical features for machine learning.

## TF-IDF Concept

**What it measures:** How important a word is to a document in a collection

**Two components:**
1. **TF (Term Frequency)**: How often a word appears in a document (high TF = common in this document)
2. **IDF (Inverse Document Frequency)**: How rare a word is across all documents (high IDF = rare, distinctive)

**Formula:** TF-IDF = TF × IDF

**Result:** Common words like "the", "and" get low scores. Distinctive words get high scores.

## Applications
- Text Classification (spam detection, sentiment analysis)
- Document Search (rank by relevance)
- Recommendation Systems (find similar documents)
- Feature Engineering (convert text to numerical features)

## Dataset

Uses EPA Fuel Economy data where text features include:
- `eng_dscr` (engine description) - free text field
- `trans_dscr` (transmission description) - free text field
- `model` (vehicle model names) - text feature

These text fields contain valuable information (turbo, hybrid, automatic) that needs numerical encoding.

## Key Considerations

- **Stop words**: Remove common words ("the", "and", "is") to reduce noise
- **Max features**: Limit vocabulary size (memory/performance)
- **N-grams**: Capture phrases ("manual transmission")
- **Sparse matrices**: TF-IDF produces sparse matrices (most values are zero)
- **Comparison**: TF-IDF (simple, fast) vs Word2Vec (semantic) vs Transformers (SOTA, heavy)

**TFIDF**

*Frequency-inverse document frequency (TFIDF)* is a numerical statistic that is intended to reflect how important a word is to a document in a collection or corpus. It is often used as a feature for text classification.

In [ ]:
import sys, pathlib
_vsc_nb = globals().get('__vsc_ipynb_file__')
_nb_dir = pathlib.Path(_vsc_nb).parent if _vsc_nb else pathlib.Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import pandas as pd
import numpy as np
from utils import (
    load_fuel_economy_data,
    debug_transformer,
    combine_str_cols_transformer,
    TextPipeline,
    CAT_COLS, LOW_CARDINALITY_COLS, HIGH_CARDINALITY_COLS,
    make_preprocessor,
)

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# create X and y
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# full pipeline: preprocessing (TF-IDF text + OHE + target encoding) → debug → scaling → regression
pipeline = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name': 'tmp_X'})),
    ('lr', LinearRegression()),
])

pipeline.fit(X_train, y_train)
print("R² on test set:", pipeline.score(X_test, y_test))